<a href="https://colab.research.google.com/github/Ferrohhh711/Ubuntu_ML-Week-4/blob/main/Ubuntu_Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1 Conceptual Foundation:

Outlier Detection Mechanisms:
Before running code, let's look at the math behind cleaning an industry dataset. Real-world agricultural records often contain anomalies (such as a single farm accidentally logging a yield 100x higher than normal due to a data entry typo). If left unchecked, these anomalies distort our regression slopes and throw off our forecasts.To locate these, your session guide requires two distinct statistical frameworks:

The Interquartile Range (IQR) Method:
We find the middle $50\%$ of our data (between the 25th percentile $Q_1$ and the 75th percentile $Q_3$). The distance between them is the IQR ($IQR = Q_3 - Q_1$). Any data point falling below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$ is flagged as an outlier. It is highly robust because it is not influenced by extreme values.The Z-Score Method: This measures how many standard deviations ($\sigma$) a data point is away from the dataset mean ($\mu$).$$Z = \frac{X - \mu}{\sigma}$$A standard rule of thumb is that any point with an absolute Z-score greater than 3 is a structural statistical anomaly.

In [1]:
import pandas as pd
import numpy as np

# 1. LOAD THE SOURCE DATASET
df = pd.read_excel('Wk 03 - Agricultural Yields Rainfall Data.xlsx')

print("=== PHASE 1: DATA HYGIENE & QUALITY AUDIT ===")
print(f"Dataset Shape: {df.shape[0]} rows | {df.shape[1]} columns")

# 2. ASSESS MISSING VALUES
print("\n Missing Values Per Feature:")
print(df.isnull().sum())

# 3. INTERQUARTILE RANGE (IQR) OUTLIER DETECTION FOR YIELD
Q1 = df['Yield_MT_per_Ha'].quantile(0.25)
Q3 = df['Yield_MT_per_Ha'].quantile(0.75)
IQR = Q3 - Q1
lower_bound_iqr = Q1 - 1.5 * IQR
upper_bound_iqr = Q3 + 1.5 * IQR

iqr_outliers = df[(df['Yield_MT_per_Ha'] < lower_bound_iqr) | (df['Yield_MT_per_Ha'] > upper_bound_iqr)]

# 4. Z-SCORE OUTLIER DETECTION FOR YIELD
mean_yield = df['Yield_MT_per_Ha'].mean()
std_yield = df['Yield_MT_per_Ha'].std()
df['Yield_Z_Score'] = (df['Yield_MT_per_Ha'] - mean_yield) / std_yield

z_outliers = df[df['Yield_Z_Score'].abs() > 3]

print("\n---Outlier Detection Summary ---")
print(f"IQR Method: Flagged {len(iqr_outliers)} yield anomalies outside boundary ({lower_bound_iqr:.2f} to {upper_bound_iqr:.2f})")
print(f"Z-Score Method: Flagged {len(z_outliers)} yield anomalies exceeding 3 standard deviations")

# Drop the temporary calculation column to keep our matrix pristine
df = df.drop(columns=['Yield_Z_Score'])

=== PHASE 1: DATA HYGIENE & QUALITY AUDIT ===
Dataset Shape: 10000 rows | 11 columns

 Missing Values Per Feature:
Year                0
County              0
Crop                0
Season              0
Yield_MT_per_Ha     0
Area_Ha             0
Total_Yield_MT      0
Rainfall_mm         0
Temp_C_Avg          0
Fertilizer_Kg_Ha    0
Yield_Index         0
dtype: int64

---Outlier Detection Summary ---
IQR Method: Flagged 484 yield anomalies outside boundary (-0.82 to 5.75)
Z-Score Method: Flagged 82 yield anomalies exceeding 3 standard deviations


In [2]:
print("=== PHASE 1 (CONTINUED): DISTRIBUTION & PERFORMANCE SUMMARIES ===")

# 1. DISTRIBUTION METRICS (Skews & Spreads)
yield_skew = df['Yield_MT_per_Ha'].skew()
rainfall_skew = df['Rainfall_mm'].skew()

print("---  Distribution Shape Diagnostics ---")
print(f"Yield Distribution Skewness   : {yield_skew:.4f} " + ("(Right-Skewed)" if yield_skew > 0 else "(Left-Skewed)"))
print(f"Rainfall Distribution Skewness: {rainfall_skew:.4f} " + ("(Right-Skewed)" if rainfall_skew > 0 else "(Left-Skewed)"))

# 2. IDENTIFY SEASONAL PATTERNS
print("\n---  Seasonal Yield Performance ---")
seasonal_summary = df.groupby('Season')['Yield_MT_per_Ha'].agg(['mean', 'std', 'count']).reset_index()
print(seasonal_summary.to_string(index=False))

# 3. COUNTY & CROP PERFORMANCE BENCHMARKS
print("\n--- Top 5 Counties by Average Yield ---")
county_summary = df.groupby('County')['Yield_MT_per_Ha'].mean().sort_values(ascending=False).head(5).reset_index()
print(county_summary.to_string(index=False))

print("\n--- Crop Performance Leaderboard ---")
crop_summary = df.groupby('Crop')['Yield_MT_per_Ha'].agg(['mean', 'min', 'max']).sort_values(by='mean', ascending=False).reset_index()
print(crop_summary.to_string(index=False))

=== PHASE 1 (CONTINUED): DISTRIBUTION & PERFORMANCE SUMMARIES ===
---  Distribution Shape Diagnostics ---
Yield Distribution Skewness   : 0.8276 (Right-Skewed)
Rainfall Distribution Skewness: 0.7862 (Right-Skewed)

---  Seasonal Yield Performance ---
               Season     mean      std  count
 Long Rains (Mar-Jul) 2.606454 1.469386   5000
Short Rains (Oct-Dec) 2.649714 1.498621   5000

--- Top 5 Counties by Average Yield ---
  County  Yield_MT_per_Ha
    Mwea         5.696434
Perkerra         5.538719
 Bunyala         5.500123
   Ahero         5.465451
    Hola         5.307107

--- Crop Performance Leaderboard ---
  Crop     mean  min  max
  Rice 5.501826 2.82 8.76
 Wheat 3.266516 1.66 5.58
 Maize 2.840910 1.57 5.04
   Tea 2.418689 1.17 4.12
Coffee 0.809434 0.45 1.46


In data science, raw numbers like Rainfall_mm or Fertilizer_Kg_Ha don't paint the whole picture. For instance, 4,000mm of rain might be amazing for rice but disastrous for coffee. To help our machine learning models capture these dynamics, we create engineered columns that reflect real-world agricultural principles.


Rainfall_Anomaly: Tells us if a season was unseasonably dry or wet by subtracting the county's historical average rainfall from the current season's rainfall.

Yield_per_mm_Rainfall: Measures water productivity (Yield divided by Rainfall).

Fertilizer_Efficiency: Measures nutrient return on investment (Yield divided by Fertilizer).

Rolling_3_Season_Yield & Rolling_5_Season_Yield: Captures short-term and long-term soil productivity trends by moving along historical averages.

County_Yield_Rank: Ranks a county's performance against others within the same crop, season, and year.

Rainfall_Volatility: Tracks climate instability by calculating the standard deviation of rainfall over a rolling 3-season window.

Yield_Growth_Rate: The percentage shift in yield from the previous season, tracking acceleration or decline.

In [3]:
print("===  PHASE 2: ADVANCED FEATURE ENGINEERING ===")

# 1. SORT DATA CHRONOLOGICALLY TO ENSURE ROLLING WINDOWS ARE VALID
df_eng = df.sort_values(by=['County', 'Crop', 'Season', 'Year']).copy()

# 2. COMPUTE BASE EFFICIENCY RATIOS
# We add a tiny epsilon (1e-5) to denominators to prevent division-by-zero errors
df_eng['Yield_per_mm_Rainfall'] = df_eng['Yield_MT_per_Ha'] / (df_eng['Rainfall_mm'] + 1e-5)
df_eng['Fertilizer_Efficiency'] = df_eng['Yield_MT_per_Ha'] / (df_eng['Fertilizer_Kg_Ha'] + 1e-5)

# 3. COMPUTE REGIONAL HISTORICAL CLIMATE ANOMALIES
county_rain_mean = df_eng.groupby('County')['Rainfall_mm'].transform('mean')
df_eng['Rainfall_Anomaly'] = df_eng['Rainfall_mm'] - county_rain_mean

# 4. COMPUTE ROLLING METRICS & GROWTH RATES
# Define our baseline grouping structure
geo_crop_groups = df_eng.groupby(['County', 'Crop', 'Season'])

df_eng['Rolling_3_Season_Yield'] = geo_crop_groups['Yield_MT_per_Ha'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df_eng['Rolling_5_Season_Yield'] = geo_crop_groups['Yield_MT_per_Ha'].transform(lambda x: x.rolling(5, min_periods=1).mean())
df_eng['Rainfall_Volatility']     = geo_crop_groups['Rainfall_mm'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
df_eng['Yield_Growth_Rate']       = geo_crop_groups['Yield_MT_per_Ha'].transform(lambda x: x.pct_change().fillna(0))

# 5. COMPUTE WITHIN-YEAR CROSS-SECTIONAL COUNTY PERFORMANCE RANKINGS
df_eng['County_Yield_Rank'] = df_eng.groupby(['Year', 'Crop', 'Season'])['Yield_MT_per_Ha'].rank(ascending=False)

print("\n Analytics dataset successfully engineered!")
print(f"Total columns expanded from 11 to {df_eng.shape[1]}")
print("\nSample of newly generated features:")
print(df_eng[['County', 'Crop', 'Rainfall_Anomaly', 'Rainfall_Volatility', 'Yield_Growth_Rate']].head(3))

===  PHASE 2: ADVANCED FEATURE ENGINEERING ===

 Analytics dataset successfully engineered!
Total columns expanded from 11 to 19

Sample of newly generated features:
     County  Crop  Rainfall_Anomaly  Rainfall_Volatility  Yield_Growth_Rate
156   Ahero  Rice        -109.67623             0.000000           0.000000
1222  Ahero  Rice          95.32377           144.956890           0.054409
1386  Ahero  Rice         367.32377           239.282957           0.028470


 Requirements:

 Answering two core business questions using rigorous mathematical metrics rather than simple observation:



*    Does rainfall significantly impact yield?

*    Does fertilizer significantly impact yield?




 To answer these, we run two categories of statistical tests:

 Correlation Coefficients (Pearson vs. Spearman):Pearson ($r$): Measures the strength and direction of a purely linear relationship. It ranges from $-1$ to $+1$.

 Spearman ($\rho$): Measures a monotonic relationship (whether values increase or decrease together, even if not in a straight line). It operates on rankings, making it highly robust against outliers and non-linear patterns.

 ANOVA (Analysis of Variance):Tests whether the average yield variations across different categorical crops are statistically meaningful or just random noise.

 $F$-Statistic: Compares the variance between the crop groups to the variance within the crop groups. A large $F$-statistic suggests the differences are stark and real.

 $p$-value: The probability that these results happened by pure chance. In data science, if $p < 0.05$, we reject the null hypothesis and confirm that the driver has a statistically significant impact.

In [4]:
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

print("=== PHASE 3: STATISTICAL ANALYSIS & HYPOTHESIS TESTING ===")

# 1. CORRELATION ANALYSIS (Rainfall vs Yield & Fertilizer vs Yield)
pearson_rain, p_pearson_rain = stats.pearsonr(df_eng['Rainfall_mm'], df_eng['Yield_MT_per_Ha'])
spearman_rain, p_spearman_rain = stats.spearmanr(df_eng['Rainfall_mm'], df_eng['Yield_MT_per_Ha'])

pearson_fert, p_pearson_fert = stats.pearsonr(df_eng['Fertilizer_Kg_Ha'], df_eng['Yield_MT_per_Ha'])
spearman_fert, p_spearman_fert = stats.spearmanr(df_eng['Fertilizer_Kg_Ha'], df_eng['Yield_MT_per_Ha'])

print("--- Correlation Matrices Summary ---")
print(f"Rainfall vs Yield  | Pearson r: {pearson_rain:.4f} (p: {p_pearson_rain:.2e}) | Spearman rho: {spearman_rain:.4f} (p: {p_spearman_rain:.2e})")
print(f"Fertilizer vs Yield| Pearson r: {pearson_fert:.4f} (p: {p_pearson_fert:.2e}) | Spearman rho: {spearman_fert:.4f} (p: {p_spearman_fert:.2e})")

# 2. ANOVA BY CROP TYPE
# Formula syntax: 'Target ~ C(Categorical_Feature)'
anova_model = ols('Yield_MT_per_Ha ~ C(Crop)', data=df_eng).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)

print("\n--- Crop Type ANOVA Analysis ---")
print(anova_table)

# 3. IDENTIFYING CLIMATE SENSITIVITY
print("\n--- Crop Sensitivity to Climate Variability (Yield Volatility) ---")
crop_sensitivity = df_eng.groupby('Crop')['Yield_MT_per_Ha'].std().sort_values(ascending=False).reset_index()
crop_sensitivity.columns = ['Crop', 'Yield_Standard_Deviation_Sigma']
print(crop_sensitivity.to_string(index=False))

=== PHASE 3: STATISTICAL ANALYSIS & HYPOTHESIS TESTING ===
--- Correlation Matrices Summary ---
Rainfall vs Yield  | Pearson r: 0.3777 (p: 0.00e+00) | Spearman rho: 0.6453 (p: 0.00e+00)
Fertilizer vs Yield| Pearson r: 0.7924 (p: 0.00e+00) | Spearman rho: 0.7515 (p: 0.00e+00)

--- Crop Type ANOVA Analysis ---
                sum_sq      df             F  PR(>F)
C(Crop)   18926.695306     4.0  15263.816758     0.0
Residual   3098.378384  9995.0           NaN     NaN

--- Crop Sensitivity to Climate Variability (Yield Volatility) ---
  Crop  Yield_Standard_Deviation_Sigma
  Rice                        1.021561
 Wheat                        0.632100
 Maize                        0.528994
   Tea                        0.455051
Coffee                        0.154625


Statistical Ground Truth:

Rainfall vs. Yield ($p = 0.00$): Our Pearson $r$ (0.3777) shows a modest linear trend, but your Spearman $\rho$ (0.6453) is much stronger. This mathematically confirms that while more rain generally increases crop yield, the relationship is non-linear—likely because extreme spikes in rainfall eventually hit saturation limits or cause waterlogging.

Fertilizer vs. Yield ($p = 0.00$): Both our Pearson (0.7924) and Spearman (0.7515) scores are highly dominant. This proves that nutrient baseline application is the single most powerful linear driver of productivity across the board.

The Crop ANOVA Test ($p = 0.00$): An $F$-statistic of 15,263.8 is quite massive. Since the $p$-value rounds to an absolute zero, we completely reject the null hypothesis. Crop type is a massive, statistically significant determinant of yield scale variations.

Climate Sensitivity (Yield $\sigma$): Rice displays the highest volatility ($\sigma = 1.02$), meaning its harvest yields fluctuate drastically depending on seasonal conditions, whereas Coffee ($\sigma = 0.15$) is highly stable.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=== PHASE 4: ENRICHED FORECASTING SHOOTOUT & FORWARD PROJECTIONS ===")

# 1. ENCODE EXTENDED CATEGORICAL SPACE & PREPARE CLEAN ENRICHED ARRAYS
X_enriched = pd.get_dummies(df_eng.drop(columns=['Year', 'Yield_MT_per_Ha', 'Total_Yield_MT', 'Yield_Index']), drop_first=True)
y_enriched = df_eng['Yield_MT_per_Ha']

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_enriched, y_enriched, test_size=0.20, random_state=42)

# 2. EVALUATE TRAINED MODELS
rf_enriched = RandomForestRegressor(n_estimators=100, random_state=42)
rf_enriched.fit(X_train_e, y_train_e)
pred_rf_e = rf_enriched.predict(X_test_e)

print(f"Enriched Random Forest Shootout R² Score: {r2_score(y_test_e, pred_rf_e):.4f}")
print(f"Enriched Random Forest Shootout RMSE    : {np.sqrt(mean_squared_error(y_test_e, pred_rf_e)):.4f} MT/Ha")

# 3. GENERATE THE LOCALLY VARIANT DYNAMIC 3-YEAR PROJECTIONS (2022 - 2024)
# Identify the last historic record for every localized profile
latest_records = df_eng.sort_values('Year').groupby(['County', 'Crop', 'Season']).last().reset_index()

forecast_list = []
rmse_val = np.sqrt(mean_squared_error(y_test_e, pred_rf_e))

for index, row in latest_records.iterrows():
    for forward_years in [2022, 2023, 2024]:
        # Copy latest historical state as our baseline structure
        fut_row = row.copy()
        fut_row['Year'] = forward_years

        # Introduce a realistic dynamic trend: simulate slight climate warming & volatility amplification
        if forward_years == 2023:
            fut_row['Rainfall_mm'] *= 0.95        # Simulate a mild drought year
            fut_row['Temp_C_Avg'] += 0.4
        elif forward_years == 2024:
            fut_row['Rainfall_mm'] *= 1.05        # Simulate an intensified rainy cycle
            fut_row['Temp_C_Avg'] += 0.7

        # Recalculate basic engineered variables based on simulated parameters
        fut_row['Yield_per_mm_Rainfall'] = fut_row['Yield_MT_per_Ha'] / (fut_row['Rainfall_mm'] + 1e-5)

        forecast_list.append(fut_row)

df_forecast_raw = pd.DataFrame(forecast_list)

# Align the future rows to our standard one-hot encoded matrix layout
X_future_encoded = pd.get_dummies(df_forecast_raw.drop(columns=['Year', 'Yield_MT_per_Ha', 'Total_Yield_MT', 'Yield_Index', 'County_Yield_Rank']), drop_first=True)
X_future_encoded = X_future_encoded.reindex(columns=X_enriched.columns, fill_value=0)

# Execute predictions across all county-crop segments
df_forecast_raw['Predicted_Yield_MT_per_Ha'] = rf_enriched.predict(X_future_encoded)
df_forecast_raw['Lower_CI_95%'] = df_forecast_raw['Predicted_Yield_MT_per_Ha'] - (1.96 * rmse_val)
df_forecast_raw['Upper_CI_95%'] = df_forecast_raw['Predicted_Yield_MT_per_Ha'] + (1.96 * rmse_val)

# Isolate columns for our downstream tracking
df_future_projections = df_forecast_raw[['Year', 'County', 'Crop', 'Season', 'Area_Ha', 'Rainfall_mm', 'Temp_C_Avg', 'Rainfall_Volatility', 'Predicted_Yield_MT_per_Ha', 'Lower_CI_95%', 'Upper_CI_95%']].copy()

print("\n--- Unique Forecasts Successfully Generated! ---")
print(df_future_projections[['Year', 'County', 'Crop', 'Predicted_Yield_MT_per_Ha']].head(6).to_string(index=False))

=== PHASE 4: ENRICHED FORECASTING SHOOTOUT & FORWARD PROJECTIONS ===
Enriched Random Forest Shootout R² Score: 0.9893
Enriched Random Forest Shootout RMSE    : 0.1560 MT/Ha

--- Unique Forecasts Successfully Generated! ---
 Year County Crop  Predicted_Yield_MT_per_Ha
 2022  Ahero Rice                     7.2353
 2023  Ahero Rice                     7.4422
 2024  Ahero Rice                     7.2301
 2022  Ahero Rice                     4.6300
 2023  Ahero Rice                     4.5998
 2024  Ahero Rice                     4.6371


To assess whether a county can feed its citizens, we look at supply versus demand using three structural metrics:

Food Requirement (Demand): The baseline food requirement assumes that every single person in a county consumes a standard 120 kg of staple crops per year.

$$\text{Food Requirement (MT)} = \frac{\text{Population} \times 120\text{ kg}}{1000}$$

Production Surplus / Deficit (Net Balance): This compares the actual localized agricultural output mass against the total survival demand of that county's population.

$$\text{Balance} = \text{Total Production (MT)} - \text{Food Requirement (MT)}$$

Self-Sufficiency Ratio (SSR): This measures a region's food independence.

$$\text{SSR} = \frac{\text{Total Production (MT)}}{\text{Food Requirement (MT)}}$$

An SSR > 1.0 indicates a secure, self-sufficient exporter that produces a surplus.

An SSR < 1.0 alerts us to a structurally dependent importer experiencing a Food Security Gap.

Since our dataset lacks a direct population column, we will map a standard population proxy dataset for these Kenyan counties directly into our code to calculate these metrics.

In [6]:
print("=== PHASE 5: FOOD SECURITY INTELLIGENCE EXCLUSIVES ===")

# 1. DEFINE REGIONAL POPULATION MAP PROXIES FOR TARGET COUNTIES
# (Representative population distribution profiles)
population_map = {
    'Nakuru': 2162202, 'Trans Nzoia': 990341, 'Uasin Gishu': 1163186,
    'Mwea': 610000, 'Perkerra': 240000, 'Bunyala': 120000,
    'Ahero': 350000, 'Hola': 180000, 'Bura': 150000,
    'Ruiru': 490000, 'Thika': 540000, 'Nyeri': 759164, 'Meru': 1545642
}

# 2. AGGREGATE TOTAL PREDICTED TONNAGE OVER THE FORECAST HORIZON
# Total Production = Predicted Yield per Ha * Total Hectares Cultivated
df_future_projections['Predicted_Total_Production_MT'] = (
    df_future_projections['Predicted_Yield_MT_per_Ha'] * df_future_projections['Area_Ha']
)

# Group by County to get total food metrics over the forecast horizon
county_food_intel = df_future_projections.groupby('County').agg({
    'Predicted_Total_Production_MT': 'sum',
    'Area_Ha': 'sum'
}).reset_index()

# 3. APPLY POPULATION MAP AND COMPUTE MACRO METRICS
county_food_intel['Population'] = county_food_intel['County'].map(population_map).fillna(500000)

# Annual requirement = Population * 120 kg per person / 1000 to convert to Metric Tons
# Multiplied by 3 to match the 3-year forecast aggregation block
county_food_intel['Food_Requirement_MT'] = (county_food_intel['Population'] * 120 / 1000) * 3

# Net Balance Metric
county_food_intel['Production_Surplus_Deficit_MT'] = (
    county_food_intel['Predicted_Total_Production_MT'] - county_food_intel['Food_Requirement_MT']
)

# Self Sufficiency Ratio (SSR)
county_food_intel['Self_Sufficiency_Ratio_SSR'] = (
    county_food_intel['Predicted_Total_Production_MT'] / county_food_intel['Food_Requirement_MT']
)

# Extract Food Security Gap (Deficit magnitude capped at 0 if there is a surplus)
county_food_intel['Food_Security_Gap_MT'] = county_food_intel['Production_Surplus_Deficit_MT'].apply(lambda x: abs(x) if x < 0 else 0)

print("\n--- County Food Security Assessment Table ---")
cols_to_print = ['County', 'Population', 'Predicted_Total_Production_MT', 'Food_Requirement_MT', 'Production_Surplus_Deficit_MT', 'Self_Sufficiency_Ratio_SSR']
print(county_food_intel[cols_to_print].sort_values(by='Production_Surplus_Deficit_MT').to_string(index=False))

=== PHASE 5: FOOD SECURITY INTELLIGENCE EXCLUSIVES ===

--- County Food Security Assessment Table ---
     County  Population  Predicted_Total_Production_MT  Food_Requirement_MT  Production_Surplus_Deficit_MT  Self_Sufficiency_Ratio_SSR
      Ruiru    490000.0                     55291.7901            176400.00                   -121108.2099                    0.313446
   Machakos    500000.0                    130334.2934            180000.00                    -49665.7066                    0.724079
     Kiambu    500000.0                    131442.3551            180000.00                    -48557.6449                    0.730235
      Thika    540000.0                    169813.5190            194400.00                    -24586.4810                    0.873526
    Kericho    500000.0                    191773.7231            180000.00                     11773.7231                    1.065410
     Kisumu    500000.0                    197707.3680            180000.00             

In [7]:
print("=== PHASE 6: COUNTY RISK SCORING ENGINE ===")

# 1. AGGREGATE HISTORICAL SENSITIVITIES FROM PHASE 2 DATA (df_eng)
county_trends = df_eng.groupby('County').agg({
    'Yield_Growth_Rate': 'mean',
    'Rainfall_Volatility': 'mean',
    'Temp_C_Avg': 'mean'
}).reset_index()

# Merge with the structural Food Gaps computed in Phase 5
df_risk = pd.merge(county_trends, county_food_intel[['County', 'Food_Security_Gap_MT']], on='County')

# 2. MIN-MAX NORMALIZATION (Scaling features uniformly from 0 to 1)
# Note: Lower or negative yield growth means HIGHER risk, so we invert the scaling calculation
df_risk['N_Yield_Trend'] = (df_risk['Yield_Growth_Rate'].max() - df_risk['Yield_Growth_Rate']) / (df_risk['Yield_Growth_Rate'].max() - df_risk['Yield_Growth_Rate'].min() + 1e-5)
df_risk['N_Rain_Vol']    = (df_risk['Rainfall_Volatility'] - df_risk['Rainfall_Volatility'].min()) / (df_risk['Rainfall_Volatility'].max() - df_risk['Rainfall_Volatility'].min() + 1e-5)
df_risk['N_Food_Gap']    = (df_risk['Food_Security_Gap_MT'] - df_risk['Food_Security_Gap_MT'].min()) / (df_risk['Food_Security_Gap_MT'].max() - df_risk['Food_Security_Gap_MT'].min() + 1e-5)
df_risk['N_Temp_Trend']  = (df_risk['Temp_C_Avg'] - df_risk['Temp_C_Avg'].min()) / (df_risk['Temp_C_Avg'].max() - df_risk['Temp_C_Avg'].min() + 1e-5)

# 3. COMPUTE WEIGHTED COMPOSITE RISK SCORE BASED ON GUIDE SPECIFICATIONS
df_risk['Risk_Score'] = (
    (df_risk['N_Yield_Trend'] * 0.40) +
    (df_risk['N_Rain_Vol'] * 0.30) +
    (df_risk['N_Food_Gap'] * 0.20) +
    (df_risk['N_Temp_Trend'] * 0.10)
)

# 4. ASSIGN RIGOROUS STRATEGIC RISK CATEGORIES
def assign_risk_category(score):
    if score > 0.7: return 'Critical'
    elif score > 0.5: return 'High'
    elif score > 0.3: return 'Medium'
    else: return 'Low'

df_risk['Risk_Category'] = df_risk['Risk_Score'].apply(assign_risk_category)

# Rank the dashboard to show the highest priority items at the top
df_risk_dashboard = df_risk[['County', 'Risk_Score', 'Risk_Category', 'Yield_Growth_Rate', 'Rainfall_Volatility', 'Food_Security_Gap_MT']].sort_values(by='Risk_Score', ascending=False)

print("\n---  PHASE 6 EXPECTED OUTPUT: COUNTY RISK RANKING DASHBOARD ---")
print(df_risk_dashboard.to_string(index=False))

=== PHASE 6: COUNTY RISK SCORING ENGINE ===

---  PHASE 6 EXPECTED OUTPUT: COUNTY RISK RANKING DASHBOARD ---
     County  Risk_Score Risk_Category  Yield_Growth_Rate  Rainfall_Volatility  Food_Security_Gap_MT
Uasin Gishu    0.611844          High           0.027269           524.348571                0.0000
      Busia    0.610912          High           0.024401           541.432195                0.0000
     Kisumu    0.608146          High           0.030144           613.888795                0.0000
   Laikipia    0.536471          High           0.028103           444.177163                0.0000
    Bungoma    0.529830          High           0.034776           662.145799                0.0000
    Kericho    0.529799          High           0.021266           225.783953                0.0000
       Hola    0.503838          High           0.025655           319.470727                0.0000
       Meru    0.489383        Medium           0.025403           283.231715              

Phase 7: Executive Summary & Capstone Challenge:

Our Capstone Challenge for the Ministry of Agriculture asks: "Which five counties should be prioritized for intervention over the next three years and why?"

 Ministry of Agriculture Strategic Intervention Memo

 To: The Cabinet Secretary, Ministry of Agriculture, Livestock, and Fisheries

 From: Lead Agricultural AI Intelligence Engine (Ferroh)
 Subject: High-Risk Regional Prioritization Plan (3-Year Horizon: 2026–2028)

 The Top 5 Priority Counties for Immediate Intervention:

 Based on a cross-sectional predictive model ($R^2 = 98.93\%$), rigorous hypothesis testing, and a weighted multi-criteria vulnerability index, the following five regions require immediate capital allocation, infrastructure staging, and policy intervention:





Deep-Dive Statistical Justifications;

1. Uasin Gishu & 2. Busia (The Vulnerable Breadbaskets)The Evidence: Uasin Gishu and Busia exhibit dangerously low historical yield growth rates (2.7% and 2.4% respectively). Concurrently, they face massive rainfall volatility standard deviations exceeding 520mm.The Policy "Why": These regions function as critical domestic supply zones. Because they are highly vulnerable to volatile rain cycles, a minor climate shock could decimate national grain supplies long before local buffer reserves can adapt.

3. Kisumu (Climate Flashpoint)The Evidence: Kisumu shows extreme seasonal weather instability with a massive 613.89mm Rainfall Volatility index and elevated mean temperatures.The Policy "Why": The high variability in rainfall creates a cycle of localized flash flooding and sudden dry spells. This heavily degrades topsoil nutrients and compromises smallholder crop consistency.

4. Laikipia (Thermal Stress Front)The Evidence: Laikipia has broken cleanly into the High-Risk category with a score of 0.5365, driven by compounding environmental factors and a notable shift in baseline temperature metrics.The Policy "Why": Rising average temperatures speed up evapotranspiration rates. This dries out crops and makes rain-fed farming systems highly unsustainable without artificial irrigation support.

5. Ruiru (The Structural Demand Crisis)The Evidence: While Ruiru sits in the "Medium Risk" band due to more stable rainfall patterns, it faces an unmitigated Food Security Gap of 121,108.21 Metric Tons and an alarming Self-Sufficiency Ratio (SSR) of just 0.313.The Policy "Why": Ruiru is an urban population center where consumer demand drastically outpaces regional farm production area. It cannot feed itself and relies entirely on external supply lines.

 Actionable Strategic Interventions:

 Pre-Stage Climate-Resilient Seeds in the Volatility Belt: Immediately deploy drought-tolerant and early-maturing seed varieties to cooperatives in Uasin Gishu and Busia before the next planting window. This buffers production against erratic weather shifts.

 Launch Supplemental Smallholder Irrigation Systems in Kisumu & Laikipia: Shift away from completely rain-fed farming dependencies by expanding water-harvesting infrastructure and solar-powered drip-irrigation grids to counter the high rainfall volatility.

 Build Strategic Urban Grain Reserves and Supply Corridors for Ruiru: Secure dedicated supply logistics channels linking high-producing agricultural zones (like Mwea and Perkerra) directly to urban storage depots in Ruiru. This insulates vulnerable urban populations from sudden commodity price surges.